# Week 4 - Text Representation I : Classic Method (2/2)
**Notebook 2 : Word Weighting & N-Grams**

**Unstructured Data Analysis (2026-2)** · Professor: Misuk Kim · Teaching Assistant: Hojin Son

Notebook 1 built the term-document matrix, where every cell is a raw count.
This notebook implements **chapters 2 and 3** of the lecture:

| Lecture slide | What we do in code |
|---|---|
| Term Frequency (TF) | rebuild the Shakespeare tf table from the NLTK Gutenberg corpus |
| Document Frequency (DF) | count in how many documents each term appears |
| Inverse Document Frequency (IDF) | reproduce the `N = 1,000,000` table by hand |
| TF-IDF | reproduce the `Term1 … Term5` worked example and its ranking |
| TF / IDF variants, SMART notation | `sublinear_tf`, `smooth_idf`, `norm` in scikit-learn |
| N-Grams | bigram counts by hand, then `ngram_range` on the real corpus |

> **How to use this notebook in Colab**: `File ▸ Save a copy in Drive`, then run the cells from top to bottom (`Shift + Enter`).

## &nbsp;0. Setup
This cell repeats, in a compact form, what Notebook 1 built step by step:
the corpus, the tokenizer, the stop-word list and the 1,000-word feature list.

In [ ]:
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

for r in ['movie_reviews', 'gutenberg', 'punkt', 'punkt_tab', 'stopwords']:
    nltk.download(r, quiet=True)

%config InlineBackend.figure_format = 'retina'
%matplotlib inline

In [ ]:
from nltk.corpus import movie_reviews, stopwords
from nltk.tokenize import RegexpTokenizer

tokenizer = RegexpTokenizer(r"[\w']{3,}")
english_stops = set(stopwords.words('english'))

reviews = [movie_reviews.raw(fileid) for fileid in movie_reviews.fileids()]


def en_tokenizer(doc):
    return [t for t in tokenizer.tokenize(doc.lower()) if t not in english_stops]


documents = [en_tokenizer(doc) for doc in reviews]

word_count = {}
for doc in documents:
    for w in doc:
        word_count[w] = word_count.get(w, 0) + 1

word_features = sorted(word_count, key=word_count.get, reverse=True)[:1000]

print('#documents:', len(documents), ' #features:', len(word_features))
print(word_features[:10])

## &nbsp;1. Term Frequency (TF)
$\text{tf}_{t,d}$ = the number of times term $t$ occurs in document $d$.
That is exactly the matrix of Notebook 1: *the more frequently a word occurs, the more important it is*.

The lecture illustrates it with a table of Shakespeare plays. NLTK's Gutenberg sample contains three of them,
so we can rebuild that table from the actual texts.

In [ ]:
from nltk.corpus import gutenberg

print([f for f in gutenberg.fileids() if 'shakespeare' in f])

In [ ]:
plays = {
    'Julius Caesar': 'shakespeare-caesar.txt',
    'Hamlet': 'shakespeare-hamlet.txt',
    'Macbeth': 'shakespeare-macbeth.txt',
}

terms = ['antony', 'brutus', 'caesar', 'calphurnia', 'cleopatra', 'mercy', 'worser']

tf_table = pd.DataFrame(index=terms)
for title, fileid in plays.items():
    words = [w.lower() for w in gutenberg.words(fileid)]
    tf_table[title] = [words.count(t) for t in terms]

tf_table

The lecture's table is reproduced almost cell for cell - `brutus` 1 and `caesar` 2 in *Hamlet*,
`mercy` 5 in *Hamlet* and 1 in *Macbeth*, `worser` 1 in *Hamlet*. (The small differences on the
*Julius Caesar* column come from a different edition of the text.)

Two rows need a comment.

In [ ]:
words = [w.lower() for w in gutenberg.words('shakespeare-caesar.txt')]

print("'calpurnia' :", words.count('calpurnia'))     # the lecture's spelling
print("'calphurnia':", words.count('calphurnia'))    # the spelling used in this edition

* **`calpurnia` vs `calphurnia`** - Caesar's wife is spelled with *ph* in the First Folio text that NLTK ships.
  Search for the lecture's spelling and you get **0**. Spelling normalization is part of preprocessing,
  and when it is missing the column simply is not there.
* **`cleopatra` = 0 everywhere** - *Antony and Cleopatra* is not in NLTK's Gutenberg sample, so the term
  never occurs. A term with `df = 0` is not a feature at all.

And look at `caesar` in *Hamlet* and *Macbeth*: small but non-zero. A raw count cannot tell us
whether a word is *characteristic* of a document or merely *present* in it. That is what DF fixes.

## &nbsp;2. Document Frequency (DF) and Inverse Document Frequency (IDF)
$\text{df}_t$ = the number of **documents** in which term $t$ appears (not how often).

The lecture's argument:

* rare terms are more informative than frequent terms (`is`, `can`, `the`, `of`, …)
* a document containing a rare query term is very likely to be relevant
* → **give a high weight to rare terms**

$$\text{idf}_t = \log_{10}\!\left(\frac{N}{\text{df}_t}\right)$$

The logarithm "dampens" the effect: without it, a term appearing in 1 of 1,000,000 documents
would get a weight a million times larger than a term appearing in all of them.

### &nbsp;2-1. The lecture table, by hand
$N = 1{,}000{,}000$.

In [ ]:
idf_example = pd.DataFrame({
    'term': ['calpurnia', 'animal', 'sunday', 'fly', 'under', 'the'],
    'df_t': [1, 100, 1_000, 10_000, 100_000, 1_000_000],
})

N = 1_000_000
idf_example['idf_t'] = np.log10(N / idf_example['df_t'])
idf_example

Each step of 10× in `df` costs exactly 1 point of `idf`, and a term present in **every** document gets `idf = 0`:
it is multiplied away completely. A stop word is simply a word whose idf is near zero - which is why
weighting is an alternative to a stop-word list, not only a complement to it.

### &nbsp;2-2. DF and IDF on our corpus
$N = 2{,}000$ movie reviews.

In [ ]:
N = len(documents)

df_t = {}
for doc in documents:
    for w in set(doc):                       # set(): count each document only once
        df_t[w] = df_t.get(w, 0) + 1

idf_t = {w: np.log10(N / df) for w, df in df_t.items()}

table = pd.DataFrame({
    'term': word_features,
    'tf (corpus)': [word_count[w] for w in word_features],
    'df': [df_t[w] for w in word_features],
    'idf': [round(idf_t[w], 3) for w in word_features],
})

print('--- most common words (lowest idf) ---')
display(table.head(8))
print('--- rarest of our 1,000 features (highest idf) ---')
display(table.tail(8))

## &nbsp;3. TF-IDF
$$\text{TF-IDF}(w) = \underbrace{\text{tf}(w)}_{\text{frequent in THIS document}} \times
\underbrace{\log\frac{N}{\text{df}(w)}}_{\text{rare in the COLLECTION}}$$

The best-known weighting scheme in information retrieval. It increases with the number of occurrences
within a document, and with the rarity of the term in the collection.

### &nbsp;3-1. The worked example from the lecture
Five terms, three documents. **Q1: which term is the most important for Doc1? Q2: which is the least?**

In [ ]:
counts = pd.DataFrame(
    [[5, 0, 0],
     [1, 0, 0],
     [5, 5, 5],
     [3, 3, 3],
     [3, 0, 1]],
    index=['Term1', 'Term2', 'Term3', 'Term4', 'Term5'],
    columns=['Doc1', 'Doc2', 'Doc3'])

counts

In [ ]:
N_docs = counts.shape[1]

result = pd.DataFrame({'TF': counts['Doc1']})
result['DF'] = (counts > 0).sum(axis=1)                 # in how many documents does the term appear
result['IDF'] = np.log10(N_docs / result['DF'])
result['TF-IDF'] = result['TF'] * result['IDF']

result.sort_values('TF-IDF', ascending=False)

**The answer.** `Term1` wins: it is frequent in Doc1 (tf = 5) **and** appears nowhere else (df = 1).

`Term3` and `Term4` lose everything: they appear in all three documents, so `idf = log(3/3) = 0`.
Note that `Term3` has the *same* raw count as `Term1` - counting alone would have ranked them equal.

And the interesting pair: `Term5` (tf 3, df 2) beats `Term2` (tf 1, df 1), even though `Term2` is rarer.
Weighting is a **product**, not a ranking by rarity alone.

> Lecture ranking: **Term1 > Term5 > Term2 > Term3 = Term4** ✓

### &nbsp;3-2. TF-IDF with scikit-learn
Two ways in:

* `TfidfTransformer` - re-weights a count matrix you already have (useful when the counts are reused)
* `TfidfVectorizer` - goes from raw text straight to TF-IDF (`CountVectorizer` + `TfidfTransformer`)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer, TfidfVectorizer

cv = CountVectorizer(vocabulary=word_features, tokenizer=en_tokenizer, token_pattern=None)
reviews_cv = cv.fit_transform(reviews)                       # the matrix from Notebook 1

reviews_tfidf = TfidfTransformer().fit_transform(reviews_cv)  # re-weight it

tf = TfidfVectorizer(vocabulary=word_features, tokenizer=en_tokenizer, token_pattern=None)
reviews_tf = tf.fit_transform(reviews)                        # or go straight from the text

print('count  matrix:', reviews_cv.shape)
print('tf-idf matrix:', reviews_tf.shape)
print('the two routes agree:', abs(reviews_tf - reviews_tfidf).max() < 1e-10)

In [ ]:
# The same document, ranked by raw count and by TF-IDF

feature_names = np.array(tf.get_feature_names_out())
doc_id = 0
c = reviews_cv[doc_id].toarray()[0]
w = reviews_tf[doc_id].toarray()[0]

print(f"{'rank':<6}{'by raw count':<22}{'by TF-IDF'}")
for r, (a, b) in enumerate(zip(np.argsort(-c)[:10], np.argsort(-w)[:10]), 1):
    print(f'{r:<6}' + f'{feature_names[a]} ({c[a]})'.ljust(22) + f'{feature_names[b]} ({w[b]:.3f})')

**❗ scikit-learn does not use the lecture's formula exactly.**
By default `smooth_idf=True`, so scikit-learn computes

$$\text{idf}_t = \ln\!\left(\frac{1+N}{1+\text{df}_t}\right) + 1$$

The `+1` terms avoid division by zero, the final `+1` keeps a term that appears everywhere from
vanishing completely, and the logarithm is natural rather than base 10.
The *ranking* is the same as the lecture's; the *numbers* are not. Let's see it.

In [ ]:
compare = pd.DataFrame({
    'term': feature_names[:8],
    'df': [df_t[w] for w in feature_names[:8]],
    'lecture  log10(N/df)': [round(np.log10(len(documents) / df_t[w]), 3) for w in feature_names[:8]],
    'sklearn  idf_': np.round(tf.idf_[:8], 3),
})
compare

In [ ]:
# Where does sklearn's number come from?  Check the formula directly.

N_ = len(documents)
for w in feature_names[:4]:
    lecture = np.log10(N_ / df_t[w])
    sk = np.log((1 + N_) / (1 + df_t[w])) + 1
    print(f'{w:<8} df={df_t[w]:<5} lecture={lecture:.3f}   ln((1+N)/(1+df))+1={sk:.3f}   '
          f'sklearn={tf.idf_[list(feature_names).index(w)]:.3f}')

# Both rank the terms in exactly the same order - only the scale differs.
print()
print('same ranking:', list(np.argsort(tf.idf_)[:5]) ==
      list(np.argsort([np.log10(N_ / df_t[w]) for w in feature_names])[:5]))

## &nbsp;4. Variants : the SMART Notation
The lecture's summary table lists three independent choices. The red entries are the common default,
written `ltc` in SMART notation:

| | option | formula | scikit-learn |
|---|---|---|---|
| Term frequency | n (natural) | $\text{tf}_{t,d}$ | `sublinear_tf=False` (default) |
| | **l (logarithm)** | $1 + \log(\text{tf}_{t,d})$ | `sublinear_tf=True` |
| | b (boolean) | 1 if $\text{tf} > 0$ | `CountVectorizer(binary=True)` |
| Document frequency | n (no) | 1 | `use_idf=False` |
| | **t (idf)** | $\log\frac{N}{\text{df}_t}$ | `use_idf=True` (default) |
| Normalization | n (none) | 1 | `norm=None` |
| | **c (cosine)** | $1/\sqrt{w_1^2 + \dots + w_M^2}$ | `norm='l2'` (default) |

Why the logarithm? A word occurring 20 times is not 20 times more important than a word occurring once.
Why normalize? Otherwise a long document simply gets larger numbers everywhere.

In [ ]:
variants = {
    "n·t·c  (default)          ": TfidfVectorizer(vocabulary=word_features, tokenizer=en_tokenizer,
                                                  token_pattern=None),
    "l·t·c  (sublinear_tf)     ": TfidfVectorizer(vocabulary=word_features, tokenizer=en_tokenizer,
                                                  token_pattern=None, sublinear_tf=True),
    "n·n·c  (no idf)           ": TfidfVectorizer(vocabulary=word_features, tokenizer=en_tokenizer,
                                                  token_pattern=None, use_idf=False),
    "n·t·n  (no normalization) ": TfidfVectorizer(vocabulary=word_features, tokenizer=en_tokenizer,
                                                  token_pattern=None, norm=None),
}

for name, vec in variants.items():
    X = vec.fit_transform(reviews)
    row = X[0].toarray()[0]
    top = feature_names[np.argsort(-row)[:5]]
    print(f'{name} top-5: {list(top)}')

In [ ]:
# What normalization does to document length

X_none = variants["n·t·n  (no normalization) "].fit_transform(reviews)
X_l2 = variants["n·t·c  (default)          "].fit_transform(reviews)

lengths = np.array([len(d) for d in documents])
norms_none = np.sqrt(X_none.multiply(X_none).sum(axis=1)).A1
norms_l2 = np.sqrt(X_l2.multiply(X_l2).sum(axis=1)).A1

print('document length   : min %d, max %d' % (lengths.min(), lengths.max()))
print('vector norm, none : min %.2f, max %.2f  <- long documents get bigger vectors'
      % (norms_none.min(), norms_none.max()))
print('vector norm, l2   : min %.2f, max %.2f  <- every document lies on the unit sphere'
      % (norms_l2.min(), norms_l2.max()))

## &nbsp;5. N-Grams
An **n-gram** is a sequence of *n* adjacent words. Two uses in the lecture:

* **Language modeling** : use the previous $n-1$ words to predict the next one
  ($P(w_n \mid w_{n-1}, \dots)$) - *"One of the hottest topics in artificial intelligence is deep ____"*
* **Text mining** : some *phrases* are far more informative than their parts -
  `six sigma`, `supply chain management`, `big data`. So we add them as features.

### &nbsp;5-1. Bigram counts by hand
The lecture shows a bigram count matrix (`i want to eat chinese food …`). Let's build one.

In [ ]:
from nltk import bigrams
from collections import Counter

mini_corpus = [
    "i want to eat chinese food",
    "i want to eat lunch",
    "i want chinese food",
    "to eat chinese food is good",
    "i spend to eat lunch",
]

words = ['i', 'want', 'to', 'eat', 'chinese', 'food', 'lunch', 'spend']

pairs = Counter()
for sent in mini_corpus:
    pairs.update(bigrams(sent.split()))

bigram_matrix = pd.DataFrame(
    [[pairs[(a, b)] for b in words] for a in words],
    index=words, columns=words)

bigram_matrix

In [ ]:
# Read it as a language model: given 'want', what is likely to follow?

row = bigram_matrix.loc['want']
print('counts after "want":', dict(row[row > 0]))
print('P(to | want) =', round(row['to'] / row.sum(), 3))

### &nbsp;5-2. N-grams as features
`ngram_range=(1, 2)` adds every adjacent word pair as an extra column. The cost is the number of features.

In [ ]:
for ngram in [(1, 1), (1, 2), (1, 3)]:
    vec = TfidfVectorizer(token_pattern=r"[a-zA-Z']{3,}",
                          lowercase=True,
                          stop_words=list(english_stops),
                          ngram_range=ngram,
                          max_df=0.5, min_df=2)
    X = vec.fit_transform(reviews)
    print(f'ngram_range={ngram}  ->  {X.shape[1]:>7,} features')

In [ ]:
vec = TfidfVectorizer(token_pattern=r"[a-zA-Z']{3,}",
                      lowercase=True,
                      stop_words=list(english_stops),
                      ngram_range=(1, 2),
                      max_df=0.5, min_df=2)
X = vec.fit_transform(reviews)

names = vec.get_feature_names_out()
bigram_feats = [f for f in names if len(f.split()) > 1]

print('#bigram features:', len(bigram_feats))
print('samples:', bigram_feats[:10])
print()

# which bigrams carry the most weight in the whole collection?
weights = np.asarray(X.sum(axis=0)).ravel()
order = np.argsort(-weights)
top_bigrams = [names[i] for i in order if len(names[i].split()) > 1][:15]
print('highest-weight bigrams:', top_bigrams)

**🤔 Quick check**
Adding bigrams more than **triples** the number of features. Adding trigrams on top changes far less -
almost every trigram occurs in a single document and is removed by `min_df=2`.

The lecture's empirical result on the REUTERS corpus points the same way: classification accuracy was
**highest with bigram features**, and fell again for 3-grams and 4-grams. Bigrams are the usual compromise.
(We will measure that ourselves in the Document Classification session.)

## &nbsp;6. The Vector Space, Finally
With TF-IDF each document is a real-valued vector in $\mathbb{R}^{|V|}$: terms are the axes,
documents are points. Now that the vectors are `l2`-normalized, the dot product between two of them
**is** the cosine of the angle between them - the standard similarity measure for text.

$$\cos(d_1, d_2) = \frac{d_1 \cdot d_2}{\lVert d_1 \rVert \, \lVert d_2 \rVert}$$

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

sim = cosine_similarity(reviews_tf[0], reviews_tf)[0]     # first review vs. all reviews
order = np.argsort(-sim)[1:6]                             # skip position 0 (the document itself)
fileids = movie_reviews.fileids()

print('query document:', fileids[0])
print()
for rank, i in enumerate(order, 1):
    print(f'{rank}. {fileids[i]:<22} cosine={sim[i]:.3f}')

In [ ]:
# The two limitations the lecture states about this space:

print('dimensionality :', reviews_tf.shape[1], 'features for', reviews_tf.shape[0], 'documents')
print('sparseness     : %.2f%% of the cells are non-zero'
      % (100 * reviews_tf.nnz / (reviews_tf.shape[0] * reviews_tf.shape[1])))

# "Very high dimensional: need to reduce the number of features!"  -> Week 7, Dimensionality Reduction
# "Sparseness: most entries are zero"                              -> Week 6, distributed representations

### ✅ What we practiced
* **TF** : the Shakespeare table rebuilt from the Gutenberg corpus
* **DF / IDF** : `idf = log10(N/df)`, the lecture's `N = 1,000,000` table, and real values on our corpus
* **TF-IDF** : the `Term1 … Term5` example reproduced, ranking and all
* scikit-learn : `TfidfTransformer` vs `TfidfVectorizer`, and how `smooth_idf` changes the numbers
* **Variants** : `sublinear_tf` (l), `use_idf` (t), `norm` (c) - the SMART `ltc` scheme
* **N-grams** : bigram counts by hand, bigrams as features, and the cost in dimensionality
* The vector space : cosine similarity, and the two problems the lecture leaves us with

| problem | where it is solved |
|---|---|
| very high dimensional | Week 7 : Dimensionality Reduction |
| sparse, and no relation between words | Week 6 : Text Representation II, distributed representations |